# UI

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.config("spark.sql.adaptive.enabled", False).getOrCreate()
sc = spark.sparkContext

In [ ]:
sc.uiWebUrl  # http://127.0.0.1:4040

Po wykonaniu poniższej komórki:
- w zakładce Jobs zobaczymy jeden job z ID 0
- job wywołany został przez akcję collect
- składa się z 2 stage'y i 8 task'ów
- liczba stage'y wynika z zastosowania transformacji agregującej - reduceByKey
- po kliknięciu w link w kolumnie Description UI przenosi nas do widoku Joba w którym widoczne są poszczególne stage
- kliknięcie w link jednego ze stage'y przedstawia statystyki opisowe tasków

In [ ]:
sc.parallelize(range(20), 4)\
.map(lambda x: (x%2, x*x))\
.mapValues(lambda x: x-1)\
.reduceByKey(lambda x,y: x+y)\
.collect()

Po wykonaniu poniższej komórki:
- w zakładce SQL (która może się pojawić dopiero po wykonaniu kodu) zobaczymy dwa wykonane zapytania
- kliknięcie w zapytanie z ID 1 przedstawi szczegółowe informacje o nim
- na wykresie widoczny jest czas działania poszczególnych stage'y (oddzielonych blokami Exchange)
- zaczynając od góry:
- blok Scan csv zawiera informację o liczbie rekordów we wczytywanym pliku
- blok Exchange wywłany przez transformację repartition wskazuje na wolumen shuffle'owanych danych
- blok WholeStageCodegen zawiera informację o liczbie wierszy powstałych w wyniku aggregacji na poszczególnych partycjach (2 partycje x 2 unikatowe wartości w kolumnie po której pogrupowane zostały dane)
- blok Exchange wywłany przez groupBy przesyła agregaty cząstkowe pomiędzy partycjami
- blok WholeStageCodegen zawiera informację o liczbie wierszy w wyjściowych danych

In [ ]:
spark.read.csv("2017-fordgobike-tripdata.csv", header=True, inferSchema=True)\
.repartition(2)\
.selectExpr("duration_sec > 60000 as ten_mins")\
.groupBy("ten_mins").count()\
.collect()

Po wykonaniu poniższej komórki:
- w zakładce storage pojawi się informacja o tym ile pamięci zajmują skaszowane dane

In [ ]:
df = spark.read.csv("2017-fordgobike-tripdata.csv", header=True, inferSchema=True)
df.cache()
df.count()